# 9.01 InMemoryStore — thread 경계를 넘는 장기 기억

`langgraph.store.memory.InMemoryStore` 는 **thread_id 스코프 밖**에서 살아남는 key/value 저장소다.
Checkpointer가 *대화 실행 이력* 을 저장한다면, Store는 *유저/조직 단위 장기 메모리* 를 저장한다.

| 관심사 | Checkpointer | Store |
|--------|--------------|-------|
| 스코프 | `thread_id` 단위 실행 상태 | `namespace` tuple 단위 자유 KV |
| 생명주기 | 대화 한 건 | 유저/조직 전체 |
| 대표 API | `get_state` / `get_state_history` | `put` / `get` / `search` |
| 의미 | "이 대화는 어디까지 진행됐나" | "이 유저가 뭘 좋아하나" |

## 학습 목표

- `put(namespace, key, value)` / `get(namespace, key)` / `search(namespace_prefix, query=...)` 의 세 축
- `namespace` tuple 로 유저·조직·용도를 계층화하는 설계
- `IndexConfig` 를 붙여 **시맨틱 검색**(임베딩 기반)을 켜는 방법
- `create_agent(..., store=...)` 로 에이전트 툴에서 `Runtime.store` 를 꺼내 쓰는 관례
- Deep Agents 의 `/memories/` 파일 규약과 Store 가 연결되는 지점

## 언제 쓰나

- 단위 테스트·로컬 개발에서 **장기 기억**(유저 선호, 사실 기억)을 빠르게 검증
- 프로덕션 Postgres/Redis Store 로 넘어가기 전 **API 같은** 드롭인 단계
- 시맨틱 검색 기반 **메모리 회상**(RAG-like) 프로토타입

## 9.01.1 환경 설정

`InMemoryStore` 자체는 외부 서비스가 필요 없다. `.env` 의 `OPENAI_API_KEY` 는 **시맨틱 검색** 섹션(9.01.4)에서만 쓴다.
키가 없어도 9.01.2~9.01.3 까지는 그대로 실행된다.

probe 는 `InMemoryStore().put(...).get(...)` 가 통과하는지만 본다 — 외부 연결 없음.

In [ ]:
# !pip install -U langgraph langchain langchain-openai

import os
from dotenv import load_dotenv
load_dotenv(override=True)

# InMemoryStore 는 키 불필요 — 즉시 사용 가능
from langgraph.store.memory import InMemoryStore

_probe = InMemoryStore()
_probe.put(("probe",), "k", {"ok": True})
assert _probe.get(("probe",), "k").value == {"ok": True}
print("InMemoryStore reachable — 외부 서비스 없음")
print("OPENAI_API_KEY for semantic search:", bool(os.environ.get("OPENAI_API_KEY")))

## 9.01.2 기본 사용 — put · get · search

세 가지 축만 기억하면 된다.

- `store.put(namespace, key, value)` — `namespace` 는 `tuple[str, ...]`, `value` 는 JSON 직렬화 가능한 dict
- `store.get(namespace, key)` — 정확히 한 건 조회, 없으면 `None`
- `store.search(namespace_prefix, query=None, filter=None, limit=10)` — prefix 매칭 + 선택적 시맨틱 질의

반환 객체는 `Item(namespace, key, value, created_at, updated_at)` 이다.

In [ ]:
store = InMemoryStore()

ns = ("user_42", "memories")
store.put(ns, "pref_theme", {"content": "다크 모드 선호"})
store.put(ns, "pref_lang",  {"content": "응답은 한국어"})
store.put(ns, "fact_role",  {"content": "백엔드 엔지니어"})

got = store.get(ns, "pref_theme")
print("get:", got.value, "created_at=", got.created_at)

print("\nsearch (prefix 만):")
for item in store.search(ns):
    print(" -", item.key, item.value)

## 9.01.3 namespace 설계 — tuple 스코프

`namespace` 는 단순 문자열이 아니라 **튜플** 이다. prefix 매칭이 가능하므로 계층적으로 쌓아 쓴다.

```
(user_id, "memories")             → 개인 기억
(org_id, "shared", "kb")           → 조직 공유 지식 베이스
(org_id, user_id, "memories")      → 조직 내 개인
("system", "prompts")              → 전역 시스템 자원
```

`search(prefix)` 는 해당 prefix 로 시작하는 모든 namespace 를 훑는다. 조직 단위 공유 메모리를 한 번에 모으거나, 유저 개인 공간만 좁히는 조작이 자연스럽다.

In [ ]:
multi = InMemoryStore()

# 조직 공유
multi.put(("acme", "shared", "kb"), "vpn_policy", {"content": "VPN 은 Always-On"})
multi.put(("acme", "shared", "kb"), "oncall",     {"content": "on-call 은 주간 로테이션"})

# 조직 내 개인
multi.put(("acme", "user_42", "memories"), "pref_theme", {"content": "다크"})
multi.put(("acme", "user_99", "memories"), "pref_theme", {"content": "라이트"})

print("조직 KB 전체:")
for it in multi.search(("acme", "shared")):
    print(" -", it.namespace, it.key, "→", it.value["content"])

print("\nuser_42 개인 기억만:")
for it in multi.search(("acme", "user_42", "memories")):
    print(" -", it.key, "→", it.value["content"])

print("\n조직 전체 namespace 목록:")
for ns in multi.list_namespaces(prefix=("acme",)):
    print(" -", ns)

## 9.01.4 `IndexConfig` + 시맨틱 검색

`InMemoryStore(index=...)` 에 `IndexConfig` 를 넘기면 put 시 **자동으로 임베딩**이 계산되고, `search(..., query=...)` 가 코사인 유사도 순으로 정렬된 `SearchItem(..., score=...)` 을 돌려준다.

```python
IndexConfig = TypedDict("IndexConfig", {
    "dims":   int,              # 임베딩 차원 (model 과 일치)
    "embed":  Embeddings | str, # langchain Embeddings 인스턴스 또는 provider:model 문자열
    "fields": list[str] | None, # value dict 중 어느 필드를 임베딩할지
})
```

`fields=["content"]` 을 주면 `value["content"]` 만 임베딩된다. 다른 필드(예: tags, timestamp)는 인덱싱되지 않는다.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embed = OpenAIEmbeddings(model="text-embedding-3-small")  # 1536-dim

sem_store = InMemoryStore(index={
    "embed": embed,
    "dims": 1536,
    "fields": ["content"],
})

ns = ("user_42", "memories")
sem_store.put(ns, "m1", {"content": "유저는 UI 다크 모드를 좋아한다",        "tag": "pref"})
sem_store.put(ns, "m2", {"content": "응답은 한국어로 달라고 요청했다",        "tag": "pref"})
sem_store.put(ns, "m3", {"content": "최근 Postgres 인덱스 튜닝을 문의했다",  "tag": "fact"})
sem_store.put(ns, "m4", {"content": "커피보다 차를 선호한다",                "tag": "pref"})

print("q='화면 테마 취향' 으로 시맨틱 검색:")
for it in sem_store.search(ns, query="화면 테마 취향", limit=3):
    print(f"  {it.score:.3f}  {it.key}  → {it.value['content']}")

print("\nquery 없이 list 하면 score=None (prefix 스캔):")
for it in sem_store.search(ns, limit=3):
    print(" -", it.key, it.score, it.value["content"][:20])

## 9.01.5 `create_agent` 에 연결 — 툴에서 `Runtime.store` 사용

`create_agent(..., store=store)` 로 Store 를 주입하면, 툴 함수가 `Runtime` 을 받아 `runtime.store` 로 동일 인스턴스에 접근한다.
이 패턴이 유용한 이유 — 툴이 **thread 경계를 넘어** 이전 대화의 선호/사실을 회상할 수 있다.

여기서는 툴 2개를 만든다.
- `save_memory` — LLM 이 새로운 기억을 저장
- `recall_memory` — LLM 이 현재 질문과 유사한 기억을 시맨틱 검색으로 끌어옴

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.runtime import get_runtime

USER_ID = "user_42"

@tool
def save_memory(content: str) -> str:
    """지속되어야 할 유저 선호/사실을 장기 기억에 저장한다."""
    rt = get_runtime()
    key = f"mem_{abs(hash(content)) % 10_000}"
    rt.store.put((USER_ID, "memories"), key, {"content": content})
    return f"saved: {key}"

@tool
def recall_memory(query: str) -> str:
    """현재 질문과 관련된 과거 유저 기억을 시맨틱으로 회상한다."""
    rt = get_runtime()
    hits = rt.store.search((USER_ID, "memories"), query=query, limit=3)
    if not hits:
        return "(기억 없음)"
    return "\n".join(f"- {h.value['content']}" for h in hits)

agent = create_agent(
    model=ChatOpenAI(model="gpt-4.1-mini"),
    tools=[save_memory, recall_memory],
    system_prompt="유저 기억 저장·회상 에이전트. 선호/사실은 save_memory, 질문 맥락은 recall_memory 를 써라.",
    store=sem_store,  # 9.01.4 에서 만든 시맨틱 Store 를 그대로 재사용
)

# thread 1: 기억 저장
r1 = agent.invoke({"messages": [{"role": "user", "content": "참고로 나는 모니터 밝기 낮은 걸 선호해."}]})
print("[저장]", r1["messages"][-1].content)

# thread 2: 새 대화에서 회상
r2 = agent.invoke({"messages": [{"role": "user", "content": "내 화면 선호가 뭐였지?"}]})
print("\n[회상]", r2["messages"][-1].content)

## 9.01.6 Deep Agents StoreBackend 관계

Deep Agents 의 **FilesystemMiddleware** 는 가상 파일 트리를 갖는다. 그중 `/memories/` prefix 아래 파일은 자동으로 StoreBackend 로 **영속** 된다.

- `/memories/*` → `(user_id, "memories")` namespace 로 Store 에 put
- 그 외 경로(`/tmp/`, `/workspace/`) → StateBackend(휘발성)

여기서 만든 `InMemoryStore` 를 `create_deep_agent(..., store=...)` 로 그대로 넘기면, 에이전트가 파일처럼 `/memories/user_prefs.md` 를 쓰기만 해도 Store 에 자동 저장된다.

프로덕션에서는 이 인스턴스만 `PostgresStore` 로 바꾸면 된다 — namespace tuple 과 key 스키마가 같기 때문에 코드 변경이 없다. 다음 노트북(`02_postgres_store.ipynb`) 주제.

## 체크리스트

- [ ] `namespace` 는 tuple — `(user_id, "memories")` 형태로 계층화
- [ ] `search(prefix, query=...)` 는 prefix 매칭 + (있으면) 시맨틱 정렬
- [ ] `IndexConfig.fields` 로 어떤 값을 임베딩할지 명시 (기본은 value 전체)
- [ ] 툴에서는 `get_runtime().store` 로 접근, `thread_id` 와 **무관**
- [ ] 프로세스 재시작 시 휘발 → 개발·테스트 용도로만 쓰고 프로덕션은 Postgres 로 이관

## 다음

- `02_postgres_store.ipynb` — 같은 API 를 Postgres 영속 버전으로 교체
- `docs/skills/langgraph-persistence.md` — Checkpointer vs Store 스코프 분리 원리
- `04_deepagents/06_memory_and_skills.ipynb` — Deep Agents 의 `/memories/` 관례와 StoreBackend 라우팅

## 참고

- 공식: https://langchain-ai.github.io/langgraph/reference/store/
- `langgraph.store.base.IndexConfig` — 임베딩 인덱스 구성 TypedDict
- `langgraph.runtime.get_runtime()` — 툴 내부에서 Store·context 접근 헬퍼